In [ ]:
# Análisis de la Prima de Riesgo Griega: Difusión Anómala vs GARCH Bayesiano

**Autores:** Ismael Sallami Moreno y David Bacas Posadas  
**Asignatura:** Seminario de Modelización

Este cuaderno presenta el código fuente y el análisis exploratorio realizado para comparar dos enfoques de modelización sobre el diferencial del bono griego a 10 años respecto al alemán (Bund):
1.  **Enfoque de Econofísica:** Análisis de difusión anómala y leyes de potencia.
2.  **Enfoque Econométrico:** Modelo de volatilidad estocástica GARCH(1,1) con inferencia Bayesiana.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import warnings
import os

# Importaciones para el modelado Bayesiano
import pymc as pm
import arviz as az
import pytensor.tensor as pt
import pytensor 

# Configuración visual
az.style.use("arviz-darkgrid")
warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente.")

In [ ]:
## 1. Carga y Preprocesamiento de Datos

Los datos provienen de series temporales de rendimientos de bonos a 10 años. Se calcula la **Prima de Riesgo** como la diferencia entre la tasa griega y la alemana. Posteriormente, calculamos la variación diaria (retornos) en puntos básicos (bps).

In [ ]:
# Configuración de archivos
ARCHIVO_GERMANY = "germany.xlsx"
ARCHIVO_GREECE = "greece.xlsx"
KEYWORD_COL_GERMANY = "Germany"
KEYWORD_COL_GREECE = "Grecia"

def convertir_coma_a_punto(x):
    """Limpia los datos numéricos con formato europeo."""
    if pd.isna(x): return np.nan
    if isinstance(x, (int, float)): return float(x)
    s = str(x).strip().replace(',', '.') 
    try:
        return float(s)
    except ValueError:
        return np.nan

def cargar_excel(archivo, keyword):
    try:
        try:
            df = pd.read_excel(archivo)
        except:
            df = pd.read_excel(archivo, engine='openpyxl')

        col_fecha = next((c for c in df.columns if 'date' in str(c).lower() or 'fecha' in str(c).lower()), None)
        if not col_fecha: col_fecha = df.columns[0]
        
        col_dato = next((c for c in df.columns if keyword.lower() in str(c).lower()), None)
        
        df = df.rename(columns={col_fecha: 'Fecha', col_dato: 'Tasa'})
        df = df[['Fecha', 'Tasa']].dropna()
        df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce', dayfirst=True)
        df = df.dropna(subset=['Fecha'])
        df['Tasa'] = df['Tasa'].apply(convertir_coma_a_punto)
        return df.set_index('Fecha').sort_index()
    except Exception as e:
        print(f"ERROR leyendo {archivo}: {e}")
        return None

# Carga de datos
df_ger = cargar_excel(ARCHIVO_GERMANY, KEYWORD_COL_GERMANY)
df_gre = cargar_excel(ARCHIVO_GREECE, KEYWORD_COL_GREECE)

# Fusión y cálculo del spread
df_comb = df_gre.join(df_ger, how='inner', lsuffix='_GRE', rsuffix='_GER')
df_comb['Prima_Riesgo'] = df_comb['Tasa_GRE'] - df_comb['Tasa_GER']

# Cálculo de variaciones (en puntos básicos)
variacion_bps = df_comb['Prima_Riesgo'].diff().dropna() * 100
y_original = variacion_bps.values

# Reescalado para estabilidad numérica en PyMC
scale_factor = np.std(y_original)
y_scaled_raw = (y_original - np.mean(y_original)) / scale_factor
floatX = pytensor.config.floatX
y_scaled = y_scaled_raw.astype(floatX)

print(f"Datos procesados. Máximo movimiento diario: {np.max(np.abs(y_original)):.2f} bps")

In [ ]:
## 2. Modelo de Difusión (Econofísica)

Analizamos si la volatilidad sigue un paseo aleatorio normal ($\alpha \approx 0.5$, difusión clásica) o si tiene memoria y persistencia ($\alpha > 0.5$, super-difusión).

Utilizamos el **Desplazamiento Cuadrático Medio (MSPD)**:
$$\langle (\Delta x)^2 \rangle \sim \tau^\alpha$$

In [ ]:
def calcular_mspd(serie, max_tau=252):
    res = {}
    vals = serie.values
    for tau in range(1, max_tau + 1):
        displacements = (vals[tau:] - vals[:-tau])**2
        res[tau] = np.mean(displacements)
    return pd.Series(res)

def power_law(t, A, alpha): return A * (t ** alpha)

# Cálculo
mspd = calcular_mspd(df_comb['Prima_Riesgo'], max_tau=min(250, len(df_comb)//4))
x_mspd = mspd.index.values
y_mspd_real = mspd.values

# Ajuste no lineal
popt, _ = curve_fit(power_law, x_mspd[:100], y_mspd_real[:100], p0=[1, 0.5])
A_fit, alpha_fit = popt
y_mspd_pred = power_law(x_mspd, *popt)

print(f"--> Exponente Alpha Calibrado: {alpha_fit:.4f}")

# Visualización
plt.figure(figsize=(8, 5))
plt.loglog(x_mspd, y_mspd_real, 'bo', label='Datos Reales (MSPD)')
plt.loglog(x_mspd, y_mspd_pred, 'r-', label=f'Ajuste (Alpha={alpha_fit:.2f})')
plt.xlabel('Retardo Temporal (Tau)')
plt.ylabel('Desplazamiento Cuadrático Medio')
plt.title('Calibración de Difusión Anómala')
plt.legend()
plt.show()

In [ ]:
## 3. Modelo GARCH(1,1) Bayesiano

Modelamos la volatilidad condicional $\sigma_t^2$ asumiendo una distribución t-Student para los retornos (colas pesadas). La inferencia se realiza mediante algoritmos MCMC (NUTS).

Especificación:
$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

In [ ]:
with pm.Model() as garch_model:
    # 1. Priors (Débilmente informativos)
    mu = pm.Normal("mu", mu=0, sigma=1, initval=0.0)
    omega = pm.InverseGamma("omega", alpha=2.5, beta=1.0, initval=0.1)
    alpha = pm.Beta("alpha", alpha=2, beta=5, initval=0.1)
    beta = pm.Beta("beta", alpha=5, beta=2, initval=0.8)
    
    # Restricción de estacionariedad
    pm.Potential("constraint", pm.math.switch(alpha + beta < 1.0, 0, -np.inf))
    
    # Grados de libertad para t-Student
    nu = pm.Gamma("nu", alpha=2, beta=0.1, initval=3.0) 

    # 2. Definición del proceso dinámico (Scan loop)
    sigma2_0 = pt.as_tensor_variable(1.0, dtype=floatX)
    
    def garch_step(y_tm1, sigma2_tm1, omega, alpha, beta, mu):
        return omega + alpha * ((y_tm1 - mu)**2) + beta * sigma2_tm1

    sigma2_cycle, _ = pytensor.scan(
        fn=garch_step,
        sequences=[y_scaled[:-1]], 
        outputs_info=[sigma2_0], 
        non_sequences=[omega, alpha, beta, mu],
        strict=True 
    )
    
    sigma2 = pm.Deterministic("sigma2", pt.concatenate([[sigma2_0], sigma2_cycle]))
    sigma = pm.Deterministic("sigma", pt.sqrt(sigma2))
    
    # 3. Likelihood
    pm.StudentT("obs", nu=nu, mu=mu, sigma=sigma, observed=y_scaled)

    # 4. Inferencia (Sampling)
    # Nota: Usamos pocos draws por demostración. Aumentar para producción.
    print("Iniciando Muestreo MCMC...")
    trace = pm.sample(draws=100, tune=100, chains=2, target_accept=0.95)
    
    # 5. Posterior Predictive Check
    ppc = pm.sample_posterior_predictive(trace, extend_inferencedata=True)

In [ ]:
### Visualización de la Volatilidad Estimada
Recuperamos la volatilidad latente inferida por el modelo y la comparamos con los movimientos reales del mercado.

In [ ]:
# Recuperar escala original
vol_scaled = trace.posterior["sigma"].mean(dim=["chain", "draw"]).values
vol_bps = vol_scaled * scale_factor

plt.figure(figsize=(12, 6))
plt.plot(variacion_bps.index, np.abs(y_original), color='gray', alpha=0.3, label='|Retornos Reales|')
plt.plot(variacion_bps.index, vol_bps, color='darkred', lw=1.5, label='Volatilidad GARCH (Media Posterior)')
plt.title('Calibración GARCH Bayesiano: Volatilidad Latente')
plt.legend()
plt.show()

In [ ]:
## 4. Comparación de Modelos (MSE)

Calculamos el Error Cuadrático Medio para ambos enfoques.
* **MSE Difusión:** Error de ajuste a la curva teórica.
* **MSE GARCH:** Error de predicción de magnitud (volatilidad) vs realidad.

In [ ]:
# MSE Difusión
mse_difusion = np.mean((y_mspd_real - y_mspd_pred) ** 2)

# MSE GARCH (Simulación Monte Carlo vs Realidad)
y_pred_mc = ppc.posterior_predictive["obs"].values 
y_pred_abs_mean = np.mean(np.abs(y_pred_mc), axis=(0, 1)) # Media de las simulaciones
mse_garch_scaled = np.mean((np.abs(y_scaled) - y_pred_abs_mean) ** 2)
mse_garch = mse_garch_scaled * (scale_factor ** 2)

print("=== RESULTADOS FINALES ===")
print(f"MSE Modelo Difusión: {mse_difusion:.4f}")
print(f"MSE Modelo GARCH:    {mse_garch:.4f}")